In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

requirements = {'numpy': 'numpy', 'pandas': 'pandas', 'plotly': 'plotly>=5.18,<8',
                'py3Dmol': 'py3Dmol>=2.4,<3', 'nbformat': 'nbformat>=5'}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None and not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
BASE_DIR = Path('/content/drive/MyDrive/phD_interview/SOLENZ/dataToPCAplots')
PROJECTION_ROOT = BASE_DIR / 'planar_projection_v3_single_wall_results_new_dataset_1410'

PROTEIN_NAME = None  # e.g. '1CEX_FsC_4rev_aligned'; folder name under PROJECTION_ROOT
ALIGNED_PROTEIN_PDB = None

MAX_BACKGROUND_POINTS = 60000
SHOW_PLANE = True
SAVE_HTML = True


In [12]:
def projection_metadata(folder):
    path = Path(folder) / 'metadata.json'
    if path.exists():
        return json.loads(path.read_text())
    with np.load(Path(folder) / 'maps.npz', allow_pickle=False) as z:
        return json.loads(str(z['metadata_json'].item()))


def available_proteins(root):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f'Edit PROJECTION_ROOT; directory not found: {root}')
    names = []
    for folder in sorted(root.iterdir()):
        if not folder.is_dir() or not all((folder / f).is_file() for f in ('points.csv', 'maps.npz')):
            continue
        meta = projection_metadata(folder)
        kind = (meta.get('projection_request') or {}).get('kind')
        if kind == 'protein' or (kind is None and not meta.get('facet_audit')):
            names.append(folder.name)
    if not names:
        raise FileNotFoundError('No protein folders with points.csv and maps.npz were found.')
    return names


def load_saved_selection(folder):
    folder = Path(folder)
    table = pd.read_csv(folder / 'points.csv', float_precision='round_trip')
    required = {'point_id', 'x', 'y', 'z', 'u_A', 'v_A', 'depth_A', 'charge',
                'contributes_to_valid_map', 'considered_by_raster', 'in_geometric_selection'}
    if required - set(table.columns):
        raise ValueError(f'points.csv lacks {sorted(required-set(table.columns))}. Use the original projection exports.')
    ids = table['point_id'].to_numpy()
    if not np.array_equal(np.sort(ids), np.arange(len(table))):
        raise ValueError('point_id must contain each zero-based shell-point ordinal exactly once.')
    table = table.sort_values('point_id').reset_index(drop=True)
    for key in ('contributes_to_valid_map', 'considered_by_raster', 'in_geometric_selection'):
        if not table[key].isin([0, 1]).all():
            raise ValueError(f'Invalid point flag: {key}')
    xyz = table[['x', 'y', 'z']].to_numpy(float)
    numeric = table[['x', 'y', 'z', 'u_A', 'v_A', 'depth_A', 'charge']].to_numpy(float)
    if not np.isfinite(numeric).all():
        raise ValueError('Nonfinite coordinates or point descriptors in points.csv.')
    with np.load(folder / 'maps.npz', allow_pickle=False) as z:
        keys = ['valid_mask', 'charge', 'depth_A', 'u_A', 'v_A', 'source_point_id',
                'source_face_id', 'plane_origin_A', 'plane_normal', 'u_axis', 'v_axis']
        if set(keys + ['metadata_json']) - set(z.files):
            raise ValueError('maps.npz lacks the saved frame or source identities from this projection version.')
        maps = {key: z[key].copy() for key in keys}
        for key in ('mesh_faces', 'barycentric_weights'):
            if key in z.files:
                maps[key] = z[key].copy()
        meta = json.loads(str(z['metadata_json'].item()))
    if (folder / 'metadata.json').exists() and projection_metadata(folder) != meta:
        raise ValueError('metadata.json and maps.npz disagree; select an unchanged export folder.')
    if meta.get('units', {}).get('coordinates') != 'angstrom':
        raise ValueError('Expected coordinates in angstroms.')
    valid = maps['valid_mask'].astype(bool)
    shape = (len(maps['v_A']), len(maps['u_A']))
    for key in ('valid_mask', 'charge', 'depth_A', 'source_point_id', 'source_face_id'):
        if maps[key].shape != shape:
            raise ValueError(f'Invalid map shape for {key}.')
    origin = np.asarray(maps['plane_origin_A'], float)
    normal, u, v = (np.asarray(maps[key], float) for key in ('plane_normal', 'u_axis', 'v_axis'))
    frame = np.array([u, v, normal])
    if (origin.shape != (3,) or frame.shape != (3, 3) or
            not np.allclose(frame @ frame.T, np.eye(3), atol=1e-7) or
            not np.allclose(np.cross(u, v), normal, atol=1e-7)):
        raise ValueError('The saved projection frame is invalid.')
    reconstructed = (origin + table['u_A'].to_numpy()[:, None]*u
                     + table['v_A'].to_numpy()[:, None]*v
                     - table['depth_A'].to_numpy()[:, None]*normal)
    error = float(np.linalg.norm(reconstructed-xyz, axis=1).max())
    if error > 1e-5:
        raise ValueError(f'Point coordinates disagree with the saved frame (max error {error:g} Å).')
    selected = table['contributes_to_valid_map'].to_numpy(bool)
    expected = np.zeros(len(table), dtype=bool)
    method = meta['summary']['method']
    if method == 'points':
        owners = maps['source_point_id'][valid]
        if (owners.dtype.kind not in 'iu' or np.any(owners < 0) or np.any(owners >= len(table))):
            raise ValueError('Invalid source_point_id in a valid pixel.')
        expected[np.unique(owners)] = True
        expected_q = table['charge'].to_numpy()[owners]
        expected_d = table['depth_A'].to_numpy()[owners]
    elif method == 'mesh':
        if not {'mesh_faces', 'barycentric_weights'} <= set(maps):
            raise ValueError('Mesh exports must retain faces and barycentric weights.')
        faces, owners = maps['mesh_faces'], maps['source_face_id'][valid]
        if (faces.ndim != 2 or faces.shape[1] != 3 or faces.dtype.kind not in 'iu' or
                np.any(faces < 0) or np.any(faces >= len(table)) or owners.dtype.kind not in 'iu' or
                np.any(owners < 0) or np.any(owners >= len(faces))):
            raise ValueError('Invalid saved mesh/source-face identities.')
        vertices = faces[owners]
        weights = maps['barycentric_weights'][valid]
        if (weights.shape != vertices.shape or not np.isfinite(weights).all() or
                np.any(weights < -1e-7) or not np.allclose(weights.sum(axis=1), 1., atol=1e-7)):
            raise ValueError('Invalid barycentric weights.')
        expected[np.unique(vertices)] = True
        expected_q = (table['charge'].to_numpy()[vertices]*weights).sum(axis=1)
        expected_d = (table['depth_A'].to_numpy()[vertices]*weights).sum(axis=1)
    else:
        raise ValueError(f'Unknown saved projection backend: {method}')
    if not np.array_equal(selected, expected):
        raise ValueError('The saved point flags disagree with the actual map source identities.')
    if (np.any(selected & ~table['considered_by_raster'].to_numpy(bool)) or
            np.any(selected & ~table['in_geometric_selection'].to_numpy(bool))):
        raise ValueError('A contributor lies outside the saved raster/geometric selection.')
    if (not np.allclose(maps['charge'][valid], expected_q, rtol=1e-9, atol=1e-10) or
            not np.allclose(maps['depth_A'][valid], expected_d, rtol=1e-9, atol=1e-10)):
        raise ValueError('Point descriptors disagree with the saved charge/depth maps.')
    if (meta['summary']['shell_points'] != len(table) or
            meta['summary']['contributing_points'] != int(selected.sum()) or
            meta['summary']['valid_pixels'] != int(valid.sum())):
        raise ValueError('Saved summary counts disagree with the exported points/maps.')
    return dict(folder=folder, points=table, xyz=xyz, selected=selected, maps=maps,
                metadata=meta, method=method, reconstruction_error_A=error)


def selection_figure(data, max_background=60000, show_plane=True):
    if max_background is not None and (not isinstance(max_background, int) or max_background < 1):
        raise ValueError('MAX_BACKGROUND_POINTS must be a positive integer or None.')
    xyz, selected, maps = data['xyz'], data['selected'], data['maps']
    background = np.flatnonzero(~selected)
    background_total = len(background)
    if max_background is not None and len(background) > max_background:
        background = np.sort(np.random.default_rng(0).choice(background, max_background, replace=False))
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=xyz[background, 0], y=xyz[background, 1], z=xyz[background, 2],
        mode='markers', name=f'',
        marker=dict(size=1.5, color='#9ba6b2', opacity=.18), hoverinfo='skip'))
    points = data['points'].loc[selected]
    name = 'Map contributors' if data['method'] == 'points' else 'Vertices of map-contributing triangles'
    fig.add_trace(go.Scatter3d(x=xyz[selected, 0], y=xyz[selected, 1], z=xyz[selected, 2],
        mode='markers', name=f'',
        marker=dict(size=2.6, color='#079b80', opacity=1.),
        customdata=points[['point_id', 'charge', 'depth_A']].to_numpy(),
        hovertemplate='point_id=%{customdata[0]:.0f}<br>charge=%{customdata[1]:.5g}'
                      '<br>depth=%{customdata[2]:.3f} Å<br>XYZ=(%{x:.3f}, %{y:.3f}, %{z:.3f}) Å<extra></extra>'))
    origin, n, u, v = (maps[key] for key in ('plane_origin_A', 'plane_normal', 'u_axis', 'v_axis'))
    h = float(data['metadata']['settings']['pixel_size_A'])
    uu, vv = np.meshgrid([maps['u_A'][0]-h/2, maps['u_A'][-1]+h/2],
                         [maps['v_A'][0]-h/2, maps['v_A'][-1]+h/2])
    plane = origin + uu[..., None]*u + vv[..., None]*v
    fig.add_trace(go.Surface(x=plane[..., 0], y=plane[..., 1], z=plane[..., 2],
        surfacecolor=np.zeros((2, 2)), colorscale=[[0, '#eab86a'], [1, '#eab86a']],
        opacity=.16, showscale=False, name='support plane', showlegend=True,
        hoverinfo='skip', visible=True if show_plane else 'legendonly'))
    length = max(float(np.ptp(xyz, axis=0).max())*.18, 1.)
    arrow = np.array([origin, origin+length*n])
    fig.add_trace(go.Scatter3d(x=arrow[:, 0], y=arrow[:, 1], z=arrow[:, 2], mode='lines+markers',
        line=dict(color='#865318', width=5), marker=dict(size=[3, 5], color='#865318'),
        name='viewing direction', hoverinfo='skip', visible=True if show_plane else 'legendonly'))
    def camera(eye):
        eye = np.asarray(eye, float); eye = 2.2*eye/np.linalg.norm(eye)
        return dict(eye=dict(zip('xyz', eye)), up=dict(zip('xyz', v)), projection=dict(type='orthographic'))
    cameras = [('Oblique', camera(n+.8*u+.4*v)), ('Front', camera(n)), ('Side', camera(u))]
    fig.update_layout(title=dict(text=f"{data['folder'].name}", x=.02),
        height=800, margin=dict(l=0, r=0, b=0, t=120),
        scene=dict(aspectmode='data', camera=cameras[0][1],
                   xaxis=dict(title='x (A)'), yaxis=dict(title='y (A)'), zaxis=dict(title='z (A)')),
        legend=dict(orientation='h', x=0, y=1.03),
        updatemenus=[dict(type='buttons', direction='right', x=0, y=1.14,
                         buttons=[dict(label=name, method='relayout', args=[{'scene.camera': cam}])
                                  for name, cam in cameras])], template='plotly_white')
    return fig


def cartoon_view(data, pdb_path, point_radius=.18):
    import py3Dmol
    pdb_path = Path(pdb_path)
    text = pdb_path.read_text()
    view = py3Dmol.view(width=950, height=700)
    view.addModel(text, 'pdb')
    view.setStyle({'model': 0}, {'cartoon': {'color': '#b6c0cc', 'opacity': .85}})
    view.addModel()
    atoms = [dict(elem='He', x=float(x), y=float(y), z=float(z), bonds=[], bondOrder=[])
             for x, y, z in data['xyz'][data['selected']]]
    view.getModel(1).addAtoms(atoms)
    view.setStyle({'model': 1}, {'sphere': {'color': '#079b80', 'radius': point_radius}})
    view.setBackgroundColor('white')
    view.zoomTo()
    return view


In [13]:
names = available_proteins(PROJECTION_ROOT)
name = PROTEIN_NAME
if name is None:
    name = next((item for item in names if item.upper().startswith('1CEX')), names[0])

surface = load_saved_selection(PROJECTION_ROOT / name)
points = surface['points']
display(pd.DataFrame([{
    'protein': name,
    'backend': surface['method'],
    'shell_points': len(points),
    'raster_candidates': int(points['considered_by_raster'].sum()),
    'map_contributors': int(surface['selected'].sum()),
    'valid_map_pixels': int(surface['maps']['valid_mask'].sum()),
    'pixel_size_A': surface['metadata']['settings']['pixel_size_A'],
    'point_radius_A': surface['metadata']['settings']['point_radius_A'],
}]))
print('Verified: point membership, map descriptors and original XYZ coordinates.')


,protein,backend,shell_points,raster_candidates,map_contributors,valid_map_pixels,pixel_size_A,point_radius_A
0,1CEX_FsC_4rev_aligned,points,9692,2148,2017,21048,0.25,0.75


Verified: point membership, map descriptors and original XYZ coordinates.


In [14]:
fig = selection_figure(surface, MAX_BACKGROUND_POINTS, SHOW_PLANE)
fig.show(config={'displaylogo': False,
                 'toImageButtonOptions': {'format': 'png', 'filename': name+'_surface_selection', 'scale': 2}})

if SAVE_HTML:
    html_path = PROJECTION_ROOT / name / 'surface_selection_3d.html'
    fig.write_html(html_path, include_plotlyjs=True, full_html=True,
                   config={'displaylogo': False})
    print('Saved:', html_path)


Saved: /content/drive/MyDrive/phD_interview/SOLENZ/dataToPCAplots/planar_projection_v3_single_wall_results_new_dataset_1410/1CEX_FsC_4rev_aligned/surface_selection_3d.html


In [6]:
if ALIGNED_PROTEIN_PDB is not None:
    molecular_view = cartoon_view(surface, ALIGNED_PROTEIN_PDB)
    molecular_view.show()
else:
    print('Optional cartoon: set ALIGNED_PROTEIN_PDB to the aligned atomistic protein PDB.')


Optional cartoon: set ALIGNED_PROTEIN_PDB to the aligned atomistic protein PDB.


**Reading the view:** drag to rotate, scroll to zoom, click legend entries to hide/show layers; use Front / Side / Oblique for fixed views. All green contributors are shown. Only the gray background may be sampled for speed.

One shell point can supply several map pixels, so contributor and valid-pixel counts need not match. HPR candidates are not automatically map contributors: the final set also depends on z-buffer visibility and the saved masks. The support-plane origin is not presented as an inferred tunnel-exit coordinate.

The optional cartoon uses 3Dmol.js in the browser. The optional HTML contains the Plotly shell view; the cartoon is displayed separately in the notebook.